# Production feature-distribution diagnostics

Inspect the root and all-comment primary predictors before fitting 06B and 06C. Full-data DuckDB scans test completeness, ranges, and basic invariants; deterministic bounded samples support plots, correlations, and selected-versus-unselected comparisons without loading either complete choice set into memory.

Success means: no missing or non-finite model values, no invalid bounded values, non-constant predictors, plausible distribution shapes, and no unexplained near-duplicate predictors.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display

SEED = int(os.getenv("COMMENTGAP_DIAGNOSTIC_SEED", "20260824"))
SAMPLE_ROWS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_SAMPLE_ROWS", "100000"))
CORRELATION_ROWS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_CORRELATION_ROWS", "50000"))
DUCKDB_THREADS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_THREADS", "8"))
STRICT_CHECKS = os.getenv("COMMENTGAP_DIAGNOSTIC_STRICT", "1") == "1"
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
OUTPUT_ROOT = Path(
    os.getenv(
        "COMMENTGAP_FEATURE_DIAGNOSTIC_ROOT",
        "model_output/selection_2025/feature_diagnostics",
    )
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
connection = duckdb.connect()
connection.execute(f"SET threads = {DUCKDB_THREADS}")
{"sample_rows": SAMPLE_ROWS, "correlation_rows": CORRELATION_ROWS, "output": str(OUTPUT_ROOT)}

## 1. Load and validate the production feature contract

This fails early if feature assembly has not been rerun with manifest v4 and the 20 expected AQuA dimensions. The composite AQuA scores must remain outside the primary feature lists.

In [ ]:
build_state = json.loads((FEATURE_ROOT / "build_state.json").read_text())
provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
registry = json.loads((FEATURE_ROOT / "feature_manifest.json").read_text())
assert build_state["status"] == "complete"
assert provenance["watermark"] == "INFERENCE"
assert provenance["models"]["aqua"]["watermark"] == "PRODUCTION"
assert registry["version"] >= 4, "Re-run commentgap-features to create the AQuA primary-model contract"

choice_paths = {scope: FEATURE_ROOT / f"choice_set_{scope}.parquet" for scope in ("root", "all")}
model_features = {scope: list(registry["models"][scope]["features"]) for scope in choice_paths}
for scope, path in choice_paths.items():
    assert path.exists(), path
    assert len(model_features[scope]) == (40 if scope == "root" else 44)
    assert len(model_features[scope]) == len(set(model_features[scope]))
    aqua = [name for name in model_features[scope] if name.startswith("aqua_")]
    assert len(aqua) == 20 and all(name.endswith("_expected") for name in aqua)
    assert "aqua_score_expected" not in model_features[scope]
    available = set(pq.ParquetFile(path).schema_arrow.names)
    missing = set(model_features[scope]) - available
    assert not missing, (scope, sorted(missing))

contract = pd.DataFrame(
    [
        {
            "scope": scope,
            "rows": pq.ParquetFile(path).metadata.num_rows,
            "model_features": len(model_features[scope]),
            "aqua_expected_features": sum(name.startswith("aqua_") for name in model_features[scope]),
        }
        for scope, path in choice_paths.items()
    ]
)
display(contract)

## 2. Full-data distribution summaries

One aggregate query per scope scans every candidate row. Approximate quantiles are sufficient for diagnostics and avoid retaining raw values. Range failures, missing values, non-finite values, and constant predictors are treated as critical. Heavy-tail flags are informational because several count-derived variables are intentionally right-skewed even after `log1p`.

In [ ]:
QUANTILES = (0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999)
BINARY_FEATURES = {
    "url_present", "vienna_overnight", "vienna_weekday_shoulder_evening",
    "vienna_weekend_day_evening", "is_reply",
}
PROBABILITY_FEATURES = {"sentiment_positive", "sentiment_negative", "toxicity_probability"}
BOUNDED_RANGES = {
    **{name: (0.0, 1.0) for name in BINARY_FEATURES | PROBABILITY_FEATURES},
    "article_similarity_top3": (-1.0, 1.0),
}

def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

def expected_range(name: str) -> tuple[float, float] | None:
    if name.startswith("aqua_") and name.endswith("_expected"):
        return (0.0, 3.0)
    return BOUNDED_RANGES.get(name)

def full_distribution_summary(scope: str) -> pd.DataFrame:
    expressions = ["COUNT(*) AS total_rows"]
    quantile_sql = "[" + ", ".join(str(value) for value in QUANTILES) + "]"
    for feature in model_features[scope]:
        column = quote_identifier(feature)
        numeric = f"CAST({column} AS DOUBLE)"
        finite = f"isfinite({numeric})"
        expressions.extend(
            [
                f"COUNT_IF({column} IS NULL) AS {quote_identifier(feature + '__nulls')}",
                f"COUNT_IF({column} IS NOT NULL AND NOT {finite}) AS {quote_identifier(feature + '__nonfinite')}",
                f"MIN({numeric}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__minimum')}",
                f"MAX({numeric}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__maximum')}",
                f"AVG({numeric}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__mean')}",
                f"STDDEV_SAMP({numeric}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__sd')}",
                f"APPROX_COUNT_DISTINCT({numeric}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__distinct')}",
                f"APPROX_QUANTILE({numeric}, {quantile_sql}) FILTER (WHERE {finite}) AS {quote_identifier(feature + '__quantiles')}",
            ]
        )
    query = "SELECT\n  " + ",\n  ".join(expressions) + "\nFROM read_parquet(?)"
    row = connection.execute(query, [str(choice_paths[scope])]).fetchdf().iloc[0]
    records = []
    for feature in model_features[scope]:
        raw_quantiles = row[f"{feature}__quantiles"]
        quantiles = list(raw_quantiles) if raw_quantiles is not None else [np.nan] * len(QUANTILES)
        record = {
            "scope": scope,
            "feature": feature,
            "label": registry["features"][feature]["label"],
            "rows": int(row["total_rows"]),
            "nulls": int(row[f"{feature}__nulls"]),
            "nonfinite": int(row[f"{feature}__nonfinite"]),
            "distinct_approx": int(row[f"{feature}__distinct"]),
            "minimum": row[f"{feature}__minimum"],
            "maximum": row[f"{feature}__maximum"],
            "mean": row[f"{feature}__mean"],
            "sd": row[f"{feature}__sd"],
            **{f"p{int(q * 1000):03d}": value for q, value in zip(QUANTILES, quantiles)},
        }
        bounds = expected_range(feature)
        record["expected_minimum"] = bounds[0] if bounds else np.nan
        record["expected_maximum"] = bounds[1] if bounds else np.nan
        record["range_violation"] = bool(
            bounds and (record["minimum"] < bounds[0] - 1e-8 or record["maximum"] > bounds[1] + 1e-8)
        )
        record["constant_or_near_constant"] = bool(
            record["distinct_approx"] <= 1 or not np.isfinite(record["sd"]) or record["sd"] <= 1e-12
        )
        lower_span = max(float(record["p500"] - record["p010"]), 1e-12)
        record["heavy_right_tail"] = bool(record["p990"] - record["p500"] > 20 * lower_span)
        records.append(record)
    return pd.DataFrame(records)

distribution_summary = pd.concat(
    [full_distribution_summary(scope) for scope in ("root", "all")], ignore_index=True
)
distribution_summary.to_csv(OUTPUT_ROOT / "full_distribution_summary.csv", index=False)
flags = distribution_summary[
    (distribution_summary["nulls"] > 0)
    | (distribution_summary["nonfinite"] > 0)
    | distribution_summary["range_violation"]
    | distribution_summary["constant_or_near_constant"]
    | distribution_summary["heavy_right_tail"]
]
display(flags if len(flags) else pd.DataFrame({"status": ["No distribution flags"]}))

## 3. Full-data row-level invariants

These checks target relationships that marginal summaries cannot detect: unique candidate keys, informative choice sets, sentiment components summing to one, maximum toxicity not falling below mean toxicity, and valid binary values.

In [ ]:
def invariant_checks(scope: str) -> pd.DataFrame:
    path = choice_paths[scope]
    available = set(pq.ParquetFile(path).schema_arrow.names)
    expressions = {
        "duplicate_candidate_keys": (
            "COUNT(*) - COUNT(DISTINCT struct_pack(story_id := story_id, comment_id := comment_id))"
        ),
        "uninformative_choice_rows": "COUNT_IF(n_candidates <= 1 OR n_picks <= 0 OR n_picks >= n_candidates)",
    }
    for feature in sorted(BINARY_FEATURES & set(model_features[scope])):
        column = quote_identifier(feature)
        expressions[f"invalid_binary__{feature}"] = f"COUNT_IF({column} NOT IN (0, 1) OR {column} IS NULL)"
    if {"sentiment_positive", "sentiment_negative", "sentiment_neutral"} <= available:
        expressions["invalid_sentiment_simplex"] = (
            "COUNT_IF(abs(sentiment_positive + sentiment_negative + sentiment_neutral - 1.0) > 1e-5)"
        )
    if {"toxicity_probability", "toxicity_mean_probability"} <= available:
        expressions["toxicity_mean_above_maximum"] = (
            "COUNT_IF(toxicity_mean_probability > toxicity_probability + 1e-8)"
        )
    aliases = list(expressions)
    query = "SELECT " + ", ".join(
        f"{expression} AS {quote_identifier(alias)}" for alias, expression in expressions.items()
    ) + " FROM read_parquet(?)"
    values = connection.execute(query, [str(path)]).fetchdf().iloc[0]
    return pd.DataFrame(
        {"scope": scope, "check": aliases, "failures": [int(values[name]) for name in aliases]}
    )

invariants = pd.concat([invariant_checks(scope) for scope in ("root", "all")], ignore_index=True)
invariants.to_csv(OUTPUT_ROOT / "row_invariant_checks.csv", index=False)
display(invariants)

def within_story_variation(scope: str) -> pd.DataFrame:
    span_expressions = []
    for feature in model_features[scope]:
        column = quote_identifier(feature)
        span_expressions.append(
            f"MAX(CAST({column} AS DOUBLE)) - MIN(CAST({column} AS DOUBLE)) AS {quote_identifier(feature)}"
        )
    inner = (
        "SELECT story_id, " + ", ".join(span_expressions)
        + " FROM read_parquet(?) GROUP BY story_id"
    )
    outer = "SELECT COUNT(*) AS stories, " + ", ".join(
        f"COUNT_IF({quote_identifier(feature)} > 1e-12) AS {quote_identifier(feature)}"
        for feature in model_features[scope]
    ) + f" FROM ({inner})"
    values = connection.execute(outer, [str(choice_paths[scope])]).fetchdf().iloc[0]
    stories = int(values["stories"])
    return pd.DataFrame(
        [
            {
                "scope": scope, "feature": feature, "stories": stories,
                "stories_with_variation": int(values[feature]),
                "proportion_stories_with_variation": float(values[feature]) / stories,
            }
            for feature in model_features[scope]
        ]
    )

within_story = pd.concat([within_story_variation(scope) for scope in ("root", "all")], ignore_index=True)
within_story.to_csv(OUTPUT_ROOT / "within_story_feature_variation.csv", index=False)
display(within_story.sort_values(["scope", "proportion_stories_with_variation"]).groupby("scope").head(12))

## 4. Deterministic bounded samples

Reservoir sampling is reproducible and row-bounded. It is used only for visualization and effect-size diagnostics; the preceding summaries and invariants use every row.

In [ ]:
samples = {}
identifier_columns = [
    "story_id", "comment_id", "n_candidates", "n_picks",
    "curator_selected", "audience_selected_draw_01",
]
for scope, path in choice_paths.items():
    available = set(pq.ParquetFile(path).schema_arrow.names)
    extras = [
        name for name in ("sentiment_neutral", "toxicity_mean_probability", "aqua_score_expected")
        if name in available
    ]
    columns = list(dict.fromkeys(identifier_columns + model_features[scope] + extras))
    select = ", ".join(quote_identifier(name) for name in columns)
    query = (
        f"SELECT {select} FROM read_parquet(?) "
        f"USING SAMPLE reservoir({SAMPLE_ROWS} ROWS) REPEATABLE ({SEED})"
    )
    samples[scope] = connection.execute(query, [str(path)]).fetchdf()
sample_accounting = pd.DataFrame(
    [{"scope": scope, "sample_rows": len(frame), "sample_stories": frame["story_id"].nunique()} for scope, frame in samples.items()]
)
display(sample_accounting)

## 5. Distribution plots

Plots are clipped only for display at the sampled 0.5th and 99.5th percentiles; the axis annotation reports that clipping. Binary predictors are shown without clipping. Root and all-comment AQuA distributions are overlaid on their native 0–3 scale.

In [ ]:
TEXT_NLP = {
    "log_words", "sentiment_positive", "sentiment_negative", "toxicity_probability",
    "lexdiv_length_adjusted", "reading_level_length_adjusted", "url_present",
}
AUTHOR_HISTORY = {name for name in set().union(*map(set, model_features.values())) if name.startswith("log_author_")}
REPLY_STRUCTURE = {"is_reply", "log_depth", "log_branch_prior_comments", "log_branch_comments_prev_hour"}

def feature_group(name: str) -> str:
    if name.startswith("aqua_"):
        return "aqua_expected"
    if name in TEXT_NLP:
        return "text_nlp"
    if name in AUTHOR_HISTORY:
        return "author_history"
    if name in REPLY_STRUCTURE:
        return "reply_structure"
    return "semantic_timing_activity"

def save_figure(fig: plt.Figure, stem: str) -> None:
    fig.savefig(OUTPUT_ROOT / f"{stem}.png", dpi=180, bbox_inches="tight")
    fig.savefig(OUTPUT_ROOT / f"{stem}.pdf", bbox_inches="tight")

def plot_distribution_grid(scope: str, features: list[str], stem: str) -> None:
    columns = 3
    rows = math.ceil(len(features) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.1 * rows), squeeze=False)
    for axis, feature in zip(axes.flat, features):
        values = pd.to_numeric(samples[scope][feature], errors="coerce").to_numpy(float)
        values = values[np.isfinite(values)]
        if feature in BINARY_FEATURES:
            counts = pd.Series(values).value_counts(normalize=True).sort_index()
            axis.bar(counts.index.astype(str), counts.values, color="#457b9d")
            axis.set_ylabel("Proportion")
        else:
            low, high = np.quantile(values, [0.005, 0.995])
            shown = np.clip(values, low, high) if high > low else values
            axis.hist(shown, bins=40, density=True, color="#457b9d", alpha=0.8)
            axis.axvline(np.median(values), color="#e76f51", linewidth=1.2)
            axis.set_xlabel(f"display clip [{low:.3g}, {high:.3g}]")
        axis.set_title(registry["features"][feature]["label"], fontsize=9)
    for axis in axes.flat[len(features):]:
        axis.set_visible(False)
    fig.suptitle(f"{scope.title()} candidates: {stem.replace('_', ' ')}", y=1.002)
    fig.tight_layout()
    save_figure(fig, f"{scope}_{stem}_distributions")
    plt.show()

for scope in ("root", "all"):
    for group in ("text_nlp", "semantic_timing_activity", "author_history", "reply_structure"):
        selected = [name for name in model_features[scope] if feature_group(name) == group]
        if selected:
            plot_distribution_grid(scope, selected, group)

In [ ]:
aqua_features = [name for name in model_features["root"] if feature_group(name) == "aqua_expected"]
columns = 4
rows = math.ceil(len(aqua_features) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(16, 2.8 * rows), sharex=True, squeeze=False)
for axis, feature in zip(axes.flat, aqua_features):
    for scope, color in (("root", "#457b9d"), ("all", "#e76f51")):
        values = pd.to_numeric(samples[scope][feature], errors="coerce").to_numpy(float)
        values = values[np.isfinite(values)]
        axis.hist(values, bins=np.linspace(0, 3, 31), density=True, histtype="step", linewidth=1.3, color=color, label=scope)
    axis.set_xlim(0, 3)
    axis.set_title(registry["features"][feature]["label"].replace("AQuA ", "").replace(" (raw expected ordinal score)", ""), fontsize=9)
for axis in axes.flat[len(aqua_features):]:
    axis.set_visible(False)
axes.flat[0].legend(frameon=False)
fig.suptitle("AQuA expected ordinal distributions", y=1.002)
fig.tight_layout()
save_figure(fig, "aqua_expected_distributions_root_vs_all")
plt.show()

## 6. Correlation and redundancy checks

Spearman correlations are calculated on at most 50,000 sampled rows per scope. Pairs with absolute correlation at least 0.90 are listed for investigation; a high correlation is not automatically an error, particularly among related AQuA dimensions and discussion-activity measures.

In [ ]:
high_correlation_rows = []
for scope in ("root", "all"):
    frame = samples[scope][model_features[scope]].head(CORRELATION_ROWS).apply(pd.to_numeric, errors="coerce")
    correlation = frame.corr(method="spearman")
    correlation.to_csv(OUTPUT_ROOT / f"{scope}_spearman_correlations.csv")
    mask = np.triu(np.ones_like(correlation, dtype=bool), k=1)
    fig, axis = plt.subplots(figsize=(16, 14))
    sns.heatmap(correlation, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, xticklabels=True, yticklabels=True, ax=axis)
    axis.tick_params(labelsize=6)
    axis.set_title(f"{scope.title()} candidates: sampled Spearman correlations")
    fig.tight_layout()
    save_figure(fig, f"{scope}_spearman_correlation_heatmap")
    plt.show()
    for left_index, left in enumerate(correlation.columns):
        for right in correlation.columns[left_index + 1:]:
            value = correlation.loc[left, right]
            if np.isfinite(value) and abs(value) >= 0.90:
                high_correlation_rows.append({"scope": scope, "feature_1": left, "feature_2": right, "spearman": value})
high_correlations = pd.DataFrame(high_correlation_rows).sort_values("spearman", key=abs, ascending=False) if high_correlation_rows else pd.DataFrame(columns=["scope", "feature_1", "feature_2", "spearman"])
high_correlations.to_csv(OUTPUT_ROOT / "high_correlation_pairs.csv", index=False)
display(high_correlations.head(50))

## 7. Selected-versus-unselected distribution tests

Standardized mean differences (SMD) and two-sample Kolmogorov–Smirnov statistics summarize separation for curator and audience selections. They are descriptive checks, not causal estimates or confirmatory hypothesis tests; no p-values are reported because the very large sample would make trivial differences statistically significant.

In [ ]:
def ks_statistic(left: np.ndarray, right: np.ndarray) -> float:
    left = np.sort(left[np.isfinite(left)])
    right = np.sort(right[np.isfinite(right)])
    if not len(left) or not len(right):
        return np.nan
    values = np.sort(np.unique(np.concatenate([left, right])))
    left_cdf = np.searchsorted(left, values, side="right") / len(left)
    right_cdf = np.searchsorted(right, values, side="right") / len(right)
    return float(np.max(np.abs(left_cdf - right_cdf)))

contrast_rows = []
for scope, frame in samples.items():
    for selector, outcome in (("curator", "curator_selected"), ("audience", "audience_selected_draw_01")):
        selected_mask = frame[outcome].astype(bool).to_numpy()
        for feature in model_features[scope]:
            values = pd.to_numeric(frame[feature], errors="coerce").to_numpy(float)
            selected = values[selected_mask]
            unselected = values[~selected_mask]
            pooled_sd = math.sqrt((np.nanvar(selected, ddof=1) + np.nanvar(unselected, ddof=1)) / 2)
            smd = (np.nanmean(selected) - np.nanmean(unselected)) / pooled_sd if pooled_sd > 0 else np.nan
            contrast_rows.append(
                {
                    "scope": scope, "selector": selector, "feature": feature,
                    "selected_mean": np.nanmean(selected), "unselected_mean": np.nanmean(unselected),
                    "standardized_mean_difference": smd,
                    "ks_statistic": ks_statistic(selected, unselected),
                }
            )
selection_contrasts = pd.DataFrame(contrast_rows)
selection_contrasts.to_csv(OUTPUT_ROOT / "selection_distribution_contrasts.csv", index=False)

for scope in ("root", "all"):
    subset = selection_contrasts[selection_contrasts["scope"] == scope].copy()
    order = (
        subset.groupby("feature")["standardized_mean_difference"].apply(lambda values: values.abs().max()).sort_values().index
    )
    positions = {feature: index for index, feature in enumerate(order)}
    fig, axis = plt.subplots(figsize=(10, max(8, 0.25 * len(order))))
    for selector, marker, color, offset in (("audience", "o", "#457b9d", -0.12), ("curator", "s", "#e76f51", 0.12)):
        data = subset[subset["selector"] == selector]
        y = np.array([positions[name] for name in data["feature"]]) + offset
        axis.scatter(data["standardized_mean_difference"], y, marker=marker, color=color, label=selector, s=24)
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_yticks(range(len(order)), [registry["features"][name]["label"] for name in order], fontsize=7)
    axis.set_xlabel("Selected minus unselected standardized mean difference")
    axis.set_title(f"{scope.title()} candidates: distribution separation")
    axis.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, f"{scope}_selection_standardized_mean_differences")
    plt.show()
display(selection_contrasts.sort_values("ks_statistic", ascending=False).head(30))

## 8. Final diagnostic status

Critical failures stop strict runs here, after tables and plots have been written. High correlations and heavy-tail warnings remain review items rather than automatic failures. Set `COMMENTGAP_DIAGNOSTIC_STRICT=0` to inspect a known-problematic build without raising the final assertion.

In [ ]:
critical_distribution_failures = distribution_summary[
    (distribution_summary["nulls"] > 0)
    | (distribution_summary["nonfinite"] > 0)
    | distribution_summary["range_violation"]
    | distribution_summary["constant_or_near_constant"]
]
critical_invariant_failures = invariants[invariants["failures"] > 0]
unidentified_features = within_story[within_story["stories_with_variation"] == 0]
status = {
    "status": "PASS" if critical_distribution_failures.empty and critical_invariant_failures.empty and unidentified_features.empty else "REVIEW_REQUIRED",
    "critical_distribution_failures": len(critical_distribution_failures),
    "critical_invariant_failures": len(critical_invariant_failures),
    "features_without_within_story_variation": len(unidentified_features),
    "informational_heavy_tail_flags": int(distribution_summary["heavy_right_tail"].sum()),
    "high_correlation_pairs": len(high_correlations),
    "output_root": str(OUTPUT_ROOT),
}
(OUTPUT_ROOT / "diagnostic_status.json").write_text(json.dumps(status, indent=2, sort_keys=True) + "\n")
display(status)
if STRICT_CHECKS:
    assert status["status"] == "PASS" and unidentified_features.empty, {
        "distribution": critical_distribution_failures[["scope", "feature", "nulls", "nonfinite", "range_violation", "constant_or_near_constant"]].to_dict("records"),
        "invariants": critical_invariant_failures.to_dict("records"),
        "features_without_within_story_variation": unidentified_features.to_dict("records"),
    }
status